# Заняття 11 · Demo 3 — дальність до цілі: LiDAR vs моно vs GT

**LIVE DEMO 3 — слайд 36.** Спирається на слайди «Рамки · KITTI» (23), «Проєкція · конвеєр» (34) і «Проєкція · результат» (35):
спершу z-buffer і щільність хмари з цих слайдів, далі дальність до цілі; останній розділ — до слайда «Чесне порівняння» (37). Це заготовка ДЗ 11 (слайд 49).

Дані — KITTI object, перші 200 кадрів `training/`: `python tools/fetch_kitti.py --object` з кореня репозиторію
(≈ 0,4 ГБ, кладе в `artifacts/kitti/object/training/`). Без них ноутбук працює на синтетичній сцені й про це скаже.

Ланцюг KITTI (слайд 23): `y = P2 · R0_rect · Tr_velo_to_cam · X`.

**GT дальності.** `location` у `label_2` — центр нижньої грані 3D-боксу, а LiDAR бачить поверхню, повернуту до сенсора:
для машини, що стоїть задом, різниця ≈ 2 м. Тому GT — медіанна глибина **видимої поверхні** 3D-боксу: промені з камери
через сітку пікселів 2D-боксу, перша точка входу в бокс. Об'єкти, які не можна чесно міряти, позначені статусом:
`оклюзія` (`occluded ≥ 2`), `обрізаний` (`truncated > 0.3`), `малий бокс` (< 25 px, поріг KITTI moderate), `< 5 точок`.

In [ ]:
%matplotlib inline
import sys, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import geom11 as G, kitti11 as K11, panels as P
P.lecture_style()                       # шрифт для проєктора, палітра Okabe–Ito


In [ ]:
IDX = 8
frame = P.kitti_frame(IDX) if K11.available(IDX) else P.synthetic_frame()
print(frame['source'], '| об\'єктів:', len(frame['objs']))


## z-buffer і щільність хмари

Слайд «Проєкція · конвеєр» (34): `depth[v, u] = z` з дублікатами індексів не гарантує, яка точка «переможе», — правильно брати
мінімум. Слайд «Проєкція · результат» (35): точка є лише в кількох відсотках пікселів, і фільтр `z > 0` — не формальність.

In [ ]:
def zbuffer(u, v, z, W, H):
    depth = np.full((H, W), np.inf)
    np.minimum.at(depth, (v.astype(int), u.astype(int)), z)
    return depth

# тест: дві точки в одному пікселі, далека записується останньою
d = zbuffer(np.array([5.2, 5.7]), np.array([3.1, 3.4]), np.array([4.0, 30.0]), 10, 10)
assert d[3, 5] == 4.0, d[3, 5]

H, W = frame['img'].shape[:2]
u, v, z, m = G.project_P(frame['pts'], frame['P'], W, H)
depth = zbuffer(u[m], v[m], z[m], W, H)
print(f'точок у кадрі: {m.sum()} з {len(m)}, покрито пікселів: {np.isfinite(depth).mean() * 100:.1f} %')

# без фільтра z > 0 точки позаду камери після ділення на z (від'ємне) потрапляють у кадр дзеркально
Y = np.c_[frame['pts'], np.ones(len(frame['pts']))] @ frame['P'].T
with np.errstate(divide='ignore', invalid='ignore'):
    ub, vb = Y[:, 0] / Y[:, 2], Y[:, 1] / Y[:, 2]
behind = (Y[:, 2] < -0.5) & (ub >= 0) & (ub < W) & (vb >= 0) & (vb < H)
print(f'без фільтра z > 0 у кадр потрапили б ще {behind.sum()} точок з-за спини камери')

## Дальність з LiDAR: frustum → перший щільний кластер → медіана

In [ ]:
def lidar_range(u, v, z, box, bin_m=0.5, min_pts=5):
    x0, y0, x1, y1 = box
    sel = (u >= x0) & (u <= x1) & (v >= y0) & (v <= y1)
    zz = np.sort(z[sel])
    if len(zz) < min_pts:
        return None, len(zz)
    hist, edges = np.histogram(zz, bins=np.arange(zz[0], zz[-1] + 2 * bin_m, bin_m))
    k = int(np.argmax(hist >= max(min_pts, 0.15 * hist.max())))
    cl = zz[(zz >= edges[k]) & (zz < edges[k] + 2.0)]
    return float(np.median(cl)), len(zz)


In [ ]:
rows = P.range_report(frame, lidar_range)

**Як читати таблицю.** Колонка «статус» — це причина, через яку дальність об'єкта не можна класти в метрику:
`оклюзія` (бокс значною частиною закритий ближчим об'єктом), `обрізаний`, `малий бокс`, `< 5 точок`. Пунктирний помаранчевий
бокс на картинці — така відмова; номер над боксом — рядок таблиці. Саме ці рядки в ДЗ ідуть окремо, а не викидаються мовчки.

**З даними KITTI.** Кадр 8 — звичайна вулиця: на `ok`-машинах LiDAR збігається з GT до 0.1–0.3 м на 6–31 м, а моно
помиляється на метри вже на 31 м (апріорна висота класу і ±2 px на боксі). Машини біля краю кадру — відмови: їх закриває
інша машина. Далі кадр 38 — ряд припаркованих машин: у частини з них `occluded = 1`, тобто формально `ok`, і в їхньому
боксі першим щільним кластером виявляється сусід ближче — LiDAR дає 5.6 м замість 9.4 і 23.6 м замість 44.9.

**Без даних KITTI.** На синтетичній сцені те саме видно на «авто 2»: воно на 62 % закрите ближчою машиною, кластер
«перескакує» на неї, і LiDAR дає ≈ 8.7 м замість ≈ 22.1. Моно тут випадково краще.

В обох випадках висновок однаковий: перевіряйте, що кластер покриває помітну частку боксу, і звіряйте з моно —
розбіжність у рази означає оклюзію, а не точний вимір.

In [ ]:
if K11.available(38):
    P.range_report(P.kitti_frame(38), lidar_range);

## Скільки точок лишається на цілі з дальністю

In [ ]:
z5 = P.points_vs_range()
print(f'HDL-64E на 10 Гц: менше 5 точок на людині — з ≈ {z5:.0f} м')


## Чесне порівняння: моно vs стерео vs LiDAR (слайд 37)

In [ ]:
Z = np.array([10, 20, 30, 40, 50, 80])
mono = np.hypot(0.1 / 1.75, 2 / 35) * Z          # σ_H/H і σ_h/h разом
stereo = Z**2 * 0.25 / (1000 * 0.12)
for z_, m_, s_ in zip(Z, mono, stereo):
    print(f'Z = {z_:3d} м   моно σ ≈ {m_:5.2f} м   стерео (B = 12 см) σ ≈ {s_:5.2f} м   {"моно краще" if m_ < s_ else "стерео краще"}')


## Питання

1. Для якого класу і на якій дальності моно помиляється найбільше? Чому (підказка: апріорний розмір)?
2. Що буде з `lidar_range`, якщо бокс детектора на 10 % ширший за об'єкт? Як це видно в гістограмі?
3. Додайте помилку `yaw = 1°` у `frame['P']` — для яких об'єктів дальність LiDAR перестала збігатися з GT?